Import Python modules

In [1]:
import pandas as pd
import numpy as np
import glob

Read in counts data

In [2]:
# Read in data
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)

# Convert codon_site to string to avoid issues with rows with semicolon-separated codon sites
counts_df['codon_site'] = counts_df['codon_site'].astype(str)

counts_df.head()

/tmp/ipykernel_978292/1834990010.py:2: DtypeWarning: Columns (0: codon_position, 1: codon_site) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis
0,71,A71C,A,C,PA,2,24,CAT,CCT,H,...,nonsynonymous,AC,all,PA,PA_all,2148,all,1.088920,0.0,False
1,1022,A1022C,A,C,NA,2,341,AAT,ACT,N,...,nonsynonymous,AC,N2,NA,NA_N2,1407,human,0.286425,0.0,False
2,1022,A1022C,A,C,NA,2,341,AAC,ACC,N,...,nonsynonymous,AC,N2,NA,NA_N2,1407,human,38.161336,0.0,False
3,166,A166C,A,C,NP,1,56,ATA,CTA,I,...,nonsynonymous,AC,all,NP,NP_all,1494,human,0.001339,0.0,False
4,166,A166C,A,C,NP,1,56,ATG,CTG,M,...,nonsynonymous,AC,all,NP,NP_all,1494,human,0.000000,NaN,False


Read in predicted rates from neutral model

In [3]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.202561
1,AC,HA,AAC,0.315628
2,AC,HA,AAG,0.248450
3,AC,HA,AAT,0.310136
4,AC,HA,CAA,0.179542


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [4]:
counts_cutoff = 10
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    # subset to rows with at least X actual or expected counts
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
)
actual_expected_df.to_csv('../results/actual_expected.csv', index=False)
print("Number of unique nucleotide mutations with data:", len(actual_expected_df))
actual_expected_df.head()

Number of unique nucleotide mutations with data: 133023


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis,predicted_rate,expected_count
2,1022,A1022C,A,C,NA,2,341,AAC,ACC,N,...,N2,NA,NA_N2,1407,human,38.161336,0.0,False,0.352564,13.454297
24,54,A54C,A,C,PA,3,18,GAA,GAC,E,...,all,PA,PA_all,2148,all,83.464153,0.011981191537307355,True,0.200742,16.754741
50,33,A33C,A,C,HA,3,11,CTA,CTC,L,...,H3,HA,HA_H3,1698,all,43.726737,0.22869302876845168,True,0.200893,8.784383
100,56,A56C,A,C,PA,2,19,AAG,ACG,K,...,all,PA,PA_all,2148,all,48.735568,0.0,True,0.246219,11.999636
109,1682,A1682C,A,C,HA,2,561,CAG,CCG,Q,...,H1,HA,HA_H1,1698,all,46.230860,0.043261146496815284,True,0.220216,10.180783


Compute fitness effects of single nucleotide mutations.

In [5]:
pseudo_count = 0.5
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'site', 'wt_nt', 'mut_nt', 'nt_mut',
    'mut_class'
]
nt_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )

    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
nt_fitness_df.to_csv('../results/nt_fitness_effects.csv', index=False)
print("Number of unique nt muts with estimated fitness effects:", len(nt_fitness_df))
nt_fitness_df.head()

Number of unique nt muts with estimated fitness effects: 105114


,host,subtype,segment,gene,site,wt_nt,mut_nt,nt_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,2,T,A,T2A,nonsynonymous,0,21.234339,-3.772041
1,all,H1,HA,HA,2,T,C,T2C,nonsynonymous,0,64.356839,-4.865330
2,all,H1,HA,HA,2,T,G,T2G,nonsynonymous,0,13.589567,-3.338582
3,all,H1,HA,HA,3,G,A,G3A,nonsynonymous,0,144.132521,-5.667343
4,all,H1,HA,HA,3,G,T,G3T,nonsynonymous,0,28.184176,-4.049493


Compute average fitness effects of synonymous nucleotide mutations at a given site.

In [6]:
groupby_cols = ['host', 'subtype', 'segment', 'gene', 'site']
site_syn_fitness_df = (
    nt_fitness_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'delta_fitness': 'mean'})
)
site_syn_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of sites with estimated synonymous fitness effects:", len(site_syn_fitness_df))
site_syn_fitness_df.head()

Number of sites with estimated synonymous fitness effects: 17367


,host,subtype,segment,gene,site,delta_fitness
0,all,H1,HA,HA,6,0.142179
1,all,H1,HA,HA,9,-0.198559
2,all,H1,HA,HA,13,-0.469505
3,all,H1,HA,HA,15,-0.928989
4,all,H1,HA,HA,16,0.568432


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [7]:
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 88286


,host,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,172.316697,-5.845379
1,all,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,21.234339,-3.772041
2,all,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,13.589567,-3.338582
3,all,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,64.356839,-4.865330
4,all,H1,HA,HA,10,C,C,C10C,synonymous,11,8.919245,0.199592


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [8]:
aa_fitness_df.groupby(['host', 'mut_class']).size()

host   mut_class    
all    nonsense          1950
       nonsynonymous    30063
       synonymous        6837
avian  nonsense           735
       nonsynonymous    13516
       synonymous        4781
human  nonsense          1628
       nonsynonymous    23373
       synonymous        5403
dtype: int64

In [9]:
aa_fitness_df[
    (aa_fitness_df['host'] == 'all') &
    (aa_fitness_df['mut_class'] == 'nonsynonymous')
].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2591
         H3         2476
         H5         1382
         H7           18
         H9          762
MP       all        1995
NA       N1         2209
         N2         2271
         N6          125
         N8          130
         N9           11
NP       all        2899
NS       all        1883
PA       all        3690
PB1      all        3599
PB2      all        4022
dtype: int64